In [1]:
import pandas as pd

# appears separated by tabs
df = pd.read_csv('data/Project2.csv', sep='\t')

print(f"Shape: {df.shape}")
print(df.head())

Shape: (342, 764)
                                                  id P1 P2 P3 P4 P5 P6 P7 P8  \
0  >ADM26535*A/wild duck/Korea/ESD48/2006*2006/03...  M  D  V  N  P  T  L  ?   
1  >AHY84452*A/Uganda/MUWRP-102/2009*2009/10/05*P...  M  D  V  N  P  T  L  ?   
2  >ABS70375*A/mallard/Maryland/369/2002*2002//*P...  M  D  V  N  P  T  L  ?   
3  >AHL89639*A/New York/1056/2007*2007/01/29*PB1*...  M  D  V  N  P  T  L  ?   
4  >AHZ43190*A/northern shoveler/Illinois/12OS513...  M  D  V  N  P  T  L  ?   

  P9  ... P754 P755 P756 P757 P758 P759 P760 P761 P762   Host  
0  L  ...    I    E    E    L    R    R    Q    K    ?  Avian  
1  L  ...    I    E    E    L    R    R    Q    K    ?  Human  
2  L  ...    I    E    E    L    K    R    Q    K    ?  Avian  
3  L  ...    I    E    D    L    R    R    Q    K    ?  Human  
4  L  ...    I    E    E    L    R    R    Q    K    ?  Avian  

[5 rows x 764 columns]


The id column looks like: `>ACCESSION*strain*date*gene*subtype*country*host_label`

we will split them into different columns for metadata

In [2]:
# Split on '*' after stripping the '>'

id_parts = df['id'].str.lstrip('>').str.split('*', expand=True)
id_parts.columns = ['accession', 'strain', 'date', 'gene', 'subtype', 'country', 'host_label']

df_clean = pd.concat([id_parts, df.drop(columns='id')], axis=1)

# Print Final Column Names
print("Final Columns:", df_clean.columns.tolist())

Final Columns: ['accession', 'strain', 'date', 'gene', 'subtype', 'country', 'host_label', 'P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26', 'P27', 'P28', 'P29', 'P30', 'P31', 'P32', 'P33', 'P34', 'P35', 'P36', 'P37', 'P38', 'P39', 'P40', 'P41', 'P42', 'P43', 'P44', 'P45', 'P46', 'P47', 'P48', 'P49', 'P50', 'P51', 'P52', 'P53', 'P54', 'P55', 'P56', 'P57', 'P58', 'P59', 'P60', 'P61', 'P62', 'P63', 'P64', 'P65', 'P66', 'P67', 'P68', 'P69', 'P70', 'P71', 'P72', 'P73', 'P74', 'P75', 'P76', 'P77', 'P78', 'P79', 'P80', 'P81', 'P82', 'P83', 'P84', 'P85', 'P86', 'P87', 'P88', 'P89', 'P90', 'P91', 'P92', 'P93', 'P94', 'P95', 'P96', 'P97', 'P98', 'P99', 'P100', 'P101', 'P102', 'P103', 'P104', 'P105', 'P106', 'P107', 'P108', 'P109', 'P110', 'P111', 'P112', 'P113', 'P114', 'P115', 'P116', 'P117', 'P118', 'P119', 'P120', 'P121', 'P122', 'P123', 'P124', 'P125', 'P126', 'P127', '

Now to inspect and settle the target column

In [4]:
# We now have 2 host label columns
print(f"Host Label values equal Host values: {df_clean['host_label'].equals(df_clean['Host'])}")

# We remove host_label and rename Host to host
df_clean = df_clean.drop(columns='host_label').rename(columns={'Host': 'host'})

# Check no NAs or ? in host
print(f"Host column has NAs: {df_clean['host'].isna().any()}")
print(f"Host column has '?': {(df_clean['host'] == '?').any()}")

# Check class balance
print("\n", df_clean['host'].value_counts())

Host Label values equal Host values: True
Host column has NAs: False
Host column has '?': False

 host
Avian    171
Human    171
Name: count, dtype: int64


* Binary classes, clean labels
* Perfectly balanced classes
* No missing values in decision class

# 

We notice there are a lot of "?", which we can assume to just be NA placeholders. Time for missing values check.

In [ ]:
# Replace '?' with NaN in the position columns
position_cols = [c for c in df_clean.columns if c.startswith('P')]
df_clean[position_cols] = df_clean[position_cols].replace('?', pd.NA)

# Check for total missing values in dataset
total_missing = df_clean.isna().sum().sum()
total_cells = df_clean.shape[0] * df_clean.shape[1]
missing_pct = (total_missing / total_cells) * 100
print(f"Total missing values: {total_missing} ({missing_pct:.2f}% of total cells)")

# Print all the position columns with missing values
missing_position_cols = df_clean[position_cols].isna().sum()
missing_position_cols = missing_position_cols[missing_position_cols > 0]
print("\nPosition columns with missing values:")
print(missing_position_cols)

Total missing values: 1704 (0.65% of total cells)

Position columns with missing values:
P1        1
P8      342
P539    342
P540    342
P739    341
P762    336
dtype: int64


In [6]:
# print the values table of P739 and P762
print(df_clean[['P739']].value_counts())
print(df_clean[['P762']].value_counts())

P739
G       1
Name: count, dtype: int64
P762
Q       6
Name: count, dtype: int64


Since we have 342 samples, this means we can safely remove Columns P8, P539, P540, P739 and P762 for being very problematic. P739 just has 1 value and then all NAs, meaning it contributes no information. 

And P762 has just 1 unique value beyond NAs, meaning it is also useless for prediction. It contributes 0 information, 0 entropy, 0 discernibility. Imputing the NAs with the Q values would just make it a constant source of zero information, while imputing them with anything else would be creating a column of 98% synthetic noise

In [7]:
# Remove columns P8, P539, P540, P739 and P762
cols_to_remove = ['P8', 'P539', 'P540', 'P739', 'P762']
df_clean = df_clean.drop(columns=cols_to_remove)

# Check again
total_missing = df_clean.isna().sum().sum()
total_cells = df_clean.shape[0] * df_clean.shape[1]
missing_pct = (total_missing / total_cells) * 100
print(f"Total missing values: {total_missing} ({missing_pct:.2f}% of total cells)")

position_cols = [c for c in df_clean.columns if c.startswith('P')]
missing_position_cols = df_clean[position_cols].isna().sum()
missing_position_cols = missing_position_cols[missing_position_cols > 0]
print("\nPosition columns with missing values:")
print(missing_position_cols)

Total missing values: 1 (0.00% of total cells)

Position columns with missing values:
P1    1
dtype: int64


We could just remove the problematic sample with the NA, but first we wanna check the P1 column because it might be another zero-variance column

In [8]:
# Check value counts in column P1
print(df_clean['P1'].value_counts(dropna=False))

P1
M       341
<NA>      1
Name: count, dtype: int64


Just as we thought. Same as P762 with "Q", P1 only has the "M" values, or otherwise NaN (which is likely just a non-written M). It does not contribute any meaningful information. Imputing an M would just make it a redundant column, and as such will be dropped. Other columns with single unique values will also be dropped since they are just noise that will make training take longer as it crowds rules that will then get filtered regardless due to shorter-rule priority policies. 

These columns may have biological meaning, they may even be tied to Influenza virus, but they are not useful (with our data) for discerning between Avian and Human influenza virus. The feature is constant and thus has 0 entropy, 0 mutual information, 0 variance, meaning 0 prediction information.

In [9]:
# Drop column P1
df_clean = df_clean.drop(columns=['P1'])

# Check all columns with only 1 unique value
unique_counts = df_clean.nunique()
cols_with_one_value = unique_counts[unique_counts == 1].index.tolist()
print("\nColumns with only 1 unique value:")
print(cols_with_one_value)

# Drop columns with only 1 unique value
df_clean = df_clean.drop(columns=cols_with_one_value)


Columns with only 1 unique value:
['gene', 'subtype', 'P7', 'P11', 'P14', 'P16', 'P17', 'P18', 'P19', 'P20', 'P22', 'P24', 'P25', 'P26', 'P27', 'P28', 'P29', 'P30', 'P31', 'P32', 'P33', 'P34', 'P35', 'P36', 'P37', 'P38', 'P40', 'P41', 'P43', 'P44', 'P45', 'P46', 'P47', 'P48', 'P49', 'P50', 'P51', 'P52', 'P126', 'P128', 'P129', 'P131', 'P132', 'P133', 'P134', 'P137', 'P138', 'P139', 'P140', 'P164', 'P169', 'P170', 'P174', 'P508', 'P510', 'P514', 'P515', 'P517', 'P519', 'P520', 'P521', 'P523', 'P524', 'P527', 'P531', 'P532', 'P533', 'P535', 'P536', 'P537', 'P542', 'P543', 'P544', 'P545', 'P546', 'P547', 'P548', 'P549', 'P551', 'P552', 'P570', 'P572', 'P573', 'P575', 'P578', 'P582', 'P583', 'P585', 'P586', 'P588', 'P591', 'P592', 'P593', 'P596', 'P597', 'P600', 'P601', 'P602', 'P603', 'P604', 'P605', 'P606', 'P607', 'P608', 'P609', 'P610', 'P614', 'P615', 'P616', 'P618', 'P619', 'P620', 'P623', 'P625', 'P626', 'P627', 'P628', 'P629', 'P632', 'P633', 'P634', 'P638', 'P642', 'P644', 'P652'

In [11]:
# Final Check
print(f"Final shape: {df_clean.shape}")
print(df_clean.head())

print("\nMissing values in final dataset:")
print(df_clean.isna().sum().sum())

Final shape: (342, 572)
  accession                                      strain        date  \
0  ADM26535                A/wild duck/Korea/ESD48/2006    2006/03/   
1  AHY84452                     A/Uganda/MUWRP-102/2009  2009/10/05   
2  ABS70375                 A/mallard/Maryland/369/2002      2002//   
3  AHL89639                        A/New York/1056/2007  2007/01/29   
4  AHZ43190  A/northern shoveler/Illinois/12OS5138/2012  2012/10/20   

       country P2 P3 P4 P5 P6 P9  ... P749 P750 P752 P753 P754 P756 P757 P758  \
0  South Korea  D  V  N  P  T  L  ...    K    I    S    T    I    E    L    R   
1       Uganda  D  V  N  P  T  L  ...    K    I    S    T    I    E    L    R   
2          USA  D  V  N  P  T  L  ...    K    I    S    T    I    E    L    K   
3          USA  D  V  N  P  T  L  ...    K    I    S    T    I    D    L    R   
4          USA  D  V  N  P  T  L  ...    K    I    S    T    I    E    L    R   

  P761   host  
0    K  Avian  
1    K  Human  
2    K  Avian 

Now, before saving, we will divide between the Metadata and the Aminoacid Position Data.
* Rename 'accession' column to 'id'
* Keep clean/positions.csv for the positional & target columns
* Keep clean/metadata.csv for the non-positional columns
* Keep 'id' column on both CSVs


In [17]:
# Rename accession to id
df_clean = df_clean.rename(columns={'accession': 'id'})

# Split data & metadata
position_cols = [c for c in df_clean.columns if c.startswith('P')]
metadata_cols = [c for c in df_clean.columns if not c.startswith('P')]

df_positions = df_clean[['id'] + position_cols + ['host']]
df_metadata = df_clean[metadata_cols].drop(columns='strain') # it's hard to interpret this column, so just drop it

# Final Views
print("Positions Data:")
print(df_positions.head())

print("\nMetadata:")
print(df_metadata.head())

Positions Data:
         id P2 P3 P4 P5 P6 P9 P10 P12 P13  ... P749 P750 P752 P753 P754 P756  \
0  ADM26535  D  V  N  P  T  L   F   K   V  ...    K    I    S    T    I    E   
1  AHY84452  D  V  N  P  T  L   F   K   I  ...    K    I    S    T    I    E   
2  ABS70375  D  V  N  P  T  L   F   K   V  ...    K    I    S    T    I    E   
3  AHL89639  D  V  N  P  T  L   F   K   V  ...    K    I    S    T    I    D   
4  AHZ43190  D  V  N  P  T  L   F   K   V  ...    K    I    S    T    I    E   

  P757 P758 P761   host  
0    L    R    K  Avian  
1    L    R    K  Human  
2    L    K    K  Avian  
3    L    R    K  Human  
4    L    R    K  Avian  

[5 rows x 569 columns]

Metadata:
         id        date      country   host
0  ADM26535    2006/03/  South Korea  Avian
1  AHY84452  2009/10/05       Uganda  Human
2  ABS70375      2002//          USA  Avian
3  AHL89639  2007/01/29          USA  Human
4  AHZ43190  2012/10/20          USA  Avian


In [ ]:
# Save the two CSVs
df_positions.to_csv('data/clean/positions.csv', index=False)
df_metadata.to_csv('data/clean/metadata.csv', index=False)

# Save Final Data
df_clean.to_csv('data/clean/full_data.csv', index=False)